# 03 · Phase-Shifting & Robust 2D Phase Unwrapping

**Sistem Fotonika — Digital Fringe Processing**  
**Iwan Cony Setiadi · Teknik Fisika**  
**Level:** S1 semester 6 dengan pengayaan riset · **Mode:** teori + implementasi + eksperimen numerik

> **Status data:** semua citra dan angka dalam notebook ini berasal dari **model/simulasi sintetis** dengan *ground truth*, bukan pengukuran instrumen nyata. Asumsi dan keterbatasan dibahas dalam setiap eksperimen.

**Tujuan:** menurunkan estimasi fase multi-frame, membandingkan 3/4/8-step, mengevaluasi error langkah fase, dan menguji unwrapping 2D.

## Asumsi dan pertanyaan metode


Dengan kamera diam dan pola dapat digeser fasenya secara terkendali, metode phase-shifting interferometry (PSI) memungkinkan estimasi fase per piksel tanpa mengandalkan pemisahan sideband. Namun syarat pentingnya adalah **objek tidak berubah selama rangkaian akuisisi** (atau perubahan dimodelkan) dan langkah fase diketahui akurat.

Model intensitas pada posisi piksel $\mathbf r=(x,y)$:

$$I_k(\mathbf r)=A(\mathbf r)+B(\mathbf r)\cos[\phi(\mathbf r)+\delta_k]+\eta_k(\mathbf r).$$

$\delta_k$ ialah pergeseran fase terprogram (rad); $k=0,\ldots,N-1$; $A,B$ tidak berubah selama pengambilan seluruh frame. Noise antarframe diasumsikan saling independen dan mempunyai varians identik dalam eksperimen awal. Semua data di notebook ini adalah **sintetis**.

## 1 · Derivasi least-squares N-step


Ekspansi sinusoidal:

$$I_k=A+C\cos\delta_k+S\sin\delta_k,\qquad C=B\cos\phi,\quad S=-B\sin\phi.$$

Untuk $N\ge3$ langkah fase yang terdistribusi sehingga matriks desain mempunyai rank 3, susun $\mathbf M=[1,\cos\delta_k,\sin\delta_k]$. Parameter diperoleh melalui $\hat{\boldsymbol\beta}=\mathbf M^+\mathbf I$, yaitu pseudoinverse/least squares. Kemudian

$$\hat\phi_w=\operatorname{atan2}(-\hat S,\hat C),\quad \hat B=\sqrt{\hat C^2+\hat S^2},\quad\hat A=\hat\beta_0.$$

Khusus empat langkah $\delta_k=0,\pi/2,\pi,3\pi/2$:

$$\boxed{\hat\phi_w=\operatorname{atan2}(I_3-I_1, I_0-I_2)}$$

**Perhatikan tanda:** persamaan ini hanya untuk model $\cos(\phi+\delta_k)$ dan penomoran $I_0,\ldots,I_3$ tersebut. Literatur yang memakai $\cos(\phi-\delta_k)$ memberikan tanda sinus berbeda. Mengganti tanda tanpa menjelaskan konvensi adalah kesalahan fisika.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi': 105, 'axes.grid': False,
                     'image.interpolation': 'nearest'})
print('NumPy:', np.__version__)

In [ ]:
from skimage.restoration import unwrap_phase

ny,nx=160,200
y,x=np.mgrid[:ny,:nx]
phi_true=1.0*np.sin(2*np.pi*x/nx)+0.9*np.cos(2*np.pi*y/ny)+2.6*x/nx
A=0.52+0.03*np.cos(2*np.pi*y/ny)
B=0.37*(0.85+0.15*np.sin(2*np.pi*x/nx)**2)

def phase_frames(phi,offsets,A,B,noise_sigma=0.0,seed=0):
    frames=A[None,:,:]+B[None,:,:]*np.cos(phi[None,:,:]+offsets[:,None,None])
    if noise_sigma>0:
        frames=frames+np.random.default_rng(seed).normal(0,noise_sigma,frames.shape)
    return frames

def phase_ls(frames,offsets):
    offsets=np.asarray(offsets)
    design=np.column_stack([np.ones_like(offsets),np.cos(offsets),np.sin(offsets)])
    if np.linalg.matrix_rank(design)<3:
        raise ValueError('Fase tidak dapat diestimasi: langkah fase rank-deficient.')
    coefficients=np.linalg.lstsq(design,frames.reshape(len(offsets),-1),rcond=None)[0]
    bg,cos_coeff,sin_coeff=(v.reshape(frames.shape[1:]) for v in coefficients)
    wrapped=np.arctan2(-sin_coeff,cos_coeff)
    modulation=np.hypot(cos_coeff,sin_coeff)
    residual=frames-(bg[None]+cos_coeff[None]*np.cos(offsets)[:,None,None]
                     +sin_coeff[None]*np.sin(offsets)[:,None,None])
    return wrapped,bg,modulation,np.sqrt(np.mean(residual**2,axis=0))

def circular_rmse(estimate,truth,mask=None):
    error=np.angle(np.exp(1j*(estimate-truth)))
    if mask is not None: error=error[mask]
    return float(np.sqrt(np.mean(error**2)))

steps4=np.arange(4)*np.pi/2
frames4=phase_frames(phi_true,steps4,A,B,noise_sigma=0.016,seed=4)
phase4,bg4,amp4,residual4=phase_ls(frames4,steps4)
phase4_closed=np.arctan2(frames4[3]-frames4[1],frames4[0]-frames4[2])
assert np.max(np.abs(np.angle(np.exp(1j*(phase4-phase4_closed)))))<1e-10
print(f'Circular RMSE 4-step: {circular_rmse(phase4,phi_true):.4f} rad')
fig,axs=plt.subplots(1,4,figsize=(14,3.3),constrained_layout=True)
for ax,im,title in zip(axs,[frames4[0],phase4,amp4,residual4],
  ['Fringe $I_0$','Fase wrapped [rad]','Modulasi terestimasi','RMS residual fit']):
    h=ax.imshow(im,cmap='gray' if title=='Fringe $I_0$' else 'viridis')
    ax.set_title(title);plt.colorbar(h,ax=ax)
plt.show()

## 2 · Noise vs jumlah langkah fase


Untuk noise Gaussian independen dengan ragam konstan, lebih banyak pengukuran pada langkah fase seragam meningkatkan presisi estimasi least-squares, tetapi membutuhkan durasi akuisisi lebih lama sehingga lebih sensitif terhadap gerakan objek. Perbandingan berikut menyamakan **noise per frame**, bukan total anggaran eksposur; hasilnya tidak boleh digeneralisasi ke kondisi fixed photon budget tanpa mengubah desain simulasi.

In [ ]:
step_counts=[3,4,8,12]
noise_grid=[0.003,0.015,0.04,0.08]
benchmark=[]
for noise in noise_grid:
    for num in step_counts:
        offsets=np.arange(num)*2*np.pi/num
        rmses=[]
        for rep in range(5):
            frames=phase_frames(phi_true,offsets,A,B,noise_sigma=noise,seed=100+rep)
            w,_,_,_=phase_ls(frames,offsets)
            rmses.append(circular_rmse(w,phi_true))
        benchmark.append((noise,num,np.mean(rmses),np.std(rmses,ddof=1)))
fig,ax=plt.subplots(figsize=(8.5,4))
for num in step_counts:
    d=[r for r in benchmark if r[1]==num]
    ax.errorbar([r[0] for r in d],[r[2] for r in d],yerr=[r[3] for r in d],
                marker='o',capsize=3,label=f'{num}-step')
ax.set(xlabel='σ noise per frame [unit intensitas]',ylabel='Circular RMSE fase [rad]',
       title='Monte Carlo (5 seeds per kondisi)');ax.legend();plt.show()
assert benchmark[0][2]>0

## 3 · Error sistematik: langkah fase aktual tidak sama dengan nominal


Jika piezo/SLM/proyektor tidak memberikan pergeseran fase yang tepat, rumus tertutup atau LS berbasis langkah nominal dapat menjadi *bias*. Berbeda dari noise acak, menambah jumlah piksel tidak otomatis menghilangkan error ini. Pada eksperimen nyata, lakukan kalibrasi fase atau gunakan algoritma yang menangani error langkah fase.

Di bawah, offsets aktual diperkenalkan gangguan deterministik; kemudian dibandingkan estimator yang menggunakan nominal dan offsets yang benar-benar diketahui. Ini **bukan** implementasi self-calibrating PSI.

In [ ]:
nominal=np.arange(4)*np.pi/2
actual=nominal+np.array([0.0,0.06,-0.045,0.035])
frames=phase_frames(phi_true,actual,A,B,noise_sigma=0.008,seed=27)
phase_wrong,*_=phase_ls(frames,nominal)
phase_cal,*_=phase_ls(frames,actual)
print('RMSE jika offset diasumsikan nominal:',round(circular_rmse(phase_wrong,phi_true),5),'rad')
print('RMSE jika offset aktual diketahui:',round(circular_rmse(phase_cal,phi_true),5),'rad')
assert circular_rmse(phase_cal,phi_true)<circular_rmse(phase_wrong,phi_true)
fig,ax=plt.subplots(1,2,figsize=(10,3.2),constrained_layout=True)
for a,v,title in zip(ax,[phase_wrong,phase_cal],['Asumsi langkah nominal','Langkah terkalibrasi']):
    im=a.imshow(np.angle(np.exp(1j*(v-phi_true))),cmap='coolwarm',vmin=-0.2,vmax=0.2)
    a.set_title(title+' (error fase [rad])');plt.colorbar(im,ax=a)
plt.show()

## 4 · Phase unwrapping 1D dan 2D


Operator $\arg$ menghasilkan fase **modulo $2\pi$**, sehingga data observasi adalah $\phi_w=\operatorname{wrap}(\phi)$ dengan $\operatorname{wrap}(\phi)=\arg(e^{i\phi})$. Unwrapping mencari bilangan bulat $k(x,y)$ sehingga $\phi_u=\phi_w+2\pi k$ memenuhi regularitas spasial tertentu. Proses ini tidak otomatis menyelesaikan *absolute phase ambiguity*: konstanta global $2\pi m$ tetap tidak teramati tanpa referensi.

- `np.unwrap` sepanjang satu atau dua sumbu memberi baseline sederhana berbasis jalur.
- `skimage.restoration.unwrap_phase` adalah algoritma 2D siap pakai untuk pembanding, bukan oracle.
- Saat noise, discontinuity, mask, dan gradien fase lebih besar dari $\pi$ per piksel terjadi, solusi dapat menjadi ambigu/keliru.

Bandingkan kedua algoritma pada fase sintetis dan tampilkan residual terhadap ground truth.

In [ ]:
wrapped_noisy=phase4
unwrap_rows=np.unwrap(np.unwrap(wrapped_noisy,axis=1),axis=0)
unwrap_2d=unwrap_phase(wrapped_noisy)

def align_estimate(estimate,truth,valid=None):
    if valid is None: valid=np.ones_like(truth,dtype=bool)
    return estimate-np.median((estimate-truth)[valid])

u_row=align_estimate(unwrap_rows,phi_true)
u_2d=align_estimate(unwrap_2d,phi_true)
print('RMSE 2D baseline:',np.sqrt(np.mean((u_row-phi_true)**2)),'rad')
print('RMSE 2D skimage :',np.sqrt(np.mean((u_2d-phi_true)**2)),'rad')
fig,ax=plt.subplots(1,4,figsize=(14,3.2),constrained_layout=True)
for a,z,t in zip(ax,[wrapped_noisy,u_row,u_2d,u_2d-phi_true],
                 ['Wrapped phase','np.unwrap (sumbu berurutan)','unwrap_phase 2D','Error 2D [rad]']):
    h=a.imshow(z,cmap='twilight' if t=='Wrapped phase' else 'viridis')
    a.set_title(t);plt.colorbar(h,ax=a)
plt.show()

## 5 · Uji kegagalan: undersampled phase gradient


Jika gradien fase diskret lokal melampaui $\pi$ rad/piksel, informasi wrapped dapat menghasilkan banyak medan unwrapped yang kompatibel dengan data yang sama. Fenomena ini **bukan masalah algoritma semata**: sampling fase itu sendiri tidak cukup. Gunakan contoh $\phi(x)=4x$ rad/piksel dan tunjukkan aliasing fase.

In [ ]:
line=np.arange(100)
true_fast=4.0*line
wrapped_fast=np.angle(np.exp(1j*true_fast))
recovered=np.unwrap(wrapped_fast)
fig,ax=plt.subplots(figsize=(8,3.4))
ax.plot(line,true_fast,label='Truth (4 rad/piksel)')
ax.plot(line,wrapped_fast,label='Wrapped')
ax.plot(line,recovered,label='Unwrapped yang teralias')
ax.set(xlabel='x [px]',ylabel='Fase [rad]',title='Kegagalan akibat gradien > π rad/piksel');ax.legend();plt.show()
assert np.max(np.abs(recovered-true_fast))>100

## 6 · Tantangan riset


Tugas: bandingkan 4-step dan 8-step pada **fixed total photon budget** (bukan fixed noise per frame), uji kesalahan langkah fase sebesar $\pm0.05$ rad, dan evaluasi mask modulasi rendah (misalnya $B<0.08$). Laporkan *circular RMSE*, residual fit, unwrapping success rate, serta risiko bias. Untuk unwrapping sulit, bandingkan path-following, quality-guided, dan least squares sesuai prasyaratnya; metode tersebut **tidak setara** secara matematis.

**Rujukan:** Ghiglia & Pritt (1998), *Two-Dimensional Phase Unwrapping: Theory, Algorithms, and Software*, Wiley (ISBN 9780471249351): https://www.wiley-vch.de/en/areas-interest/engineering/two-dimensional-phase-unwrapping-978-0-471-24935-1. Pendahuluan algoritma PSI: University of Arizona, https://webs.optics.arizona.edu/gsmith/PhaseShift.html.

<!-- AUDIT-RESEARCH-2026 -->

## 7 · Prediksi statistik PSI dan eksperimen *fixed photon budget*

Dari model LS dengan $N$ fase seragam, $B>0$, dan noise aditif iid ber-varians $\sigma^2$ pada **tiap frame**, pendekatan galat kecil menghasilkan

$$\operatorname{Var}(\hat\phi)\simeq\frac{2\sigma^2}{N B^2}.$$

Persamaan ini menuntut matriks desain yang ortogonal, tidak ada saturasi, tanpa kesalahan phase step dan phase wrapping yang gagal. **Noise per frame sama** bukanlah *total photon budget* yang sama. Jika anggaran fotoelektron total tetap, lebih banyak frame membagi jumlah foton ke lebih banyak exposure, sedangkan read noise terjadi tiap exposure.

Dalam sel berikut digunakan satu piksel ideal $I_k=A+B\cos(\phi+\delta_k)$, $A=0.50$, $B=0.30$, dan *expected electron count* $\mu_k=(F/N)(I_k/A)$. Observasi Poisson+read-noise dipetakan kembali ke intensitas linear. Uji Monte Carlo 1000 realisasi per kondisi membandingkan ketidakpastian fase, **tidak mengklaim generalisasi untuk kamera nyata**. Model ini menyederhanakan quantum efficiency, dark current, dan korelasi elektron.

In [ ]:
# Fixed total photon budget (single pixel): use identical counts budget F across N
F_total=5000.
A0,B0,phi0 = .5,.3,1.13
nrep=1200
rng_photon=np.random.default_rng(3122)
photon_rows=[]
for nstep in [3,4,8,12]:
    offsets=np.arange(nstep)*2*np.pi/nstep
    I_model=A0+B0*np.cos(phi0+offsets)
    expected_counts=(F_total/nstep)*I_model/A0
    for read_e in [0.,9.]:
        observed=rng_photon.poisson(expected_counts[:,None],size=(nstep,nrep))
        observed=observed+rng_photon.normal(0,read_e,(nstep,nrep))
        normalized=observed*A0/(F_total/nstep)
        X=np.column_stack([np.ones(nstep),np.cos(offsets),np.sin(offsets)])
        est=np.linalg.lstsq(X,normalized,rcond=None)[0]
        phi_hat=np.arctan2(-est[2],est[1])
        err=np.angle(np.exp(1j*(phi_hat-phi0)))
        photon_rows.append((nstep,read_e,float(np.std(err,ddof=1)),float(np.mean(err))))
print('N, read σ [e], std phase [rad], bias [rad]')
for row in photon_rows: print(row)
assert all(np.isfinite(row[2]) for row in photon_rows)
assert photon_rows[-1][2] > photon_rows[3][2] # 12-step vs 4-step: read noise accumulates for fixed F
fig,ax=plt.subplots(figsize=(7.5,3.5))
for read_e in [0.,9.]:
    rows=[r for r in photon_rows if r[1]==read_e]
    ax.plot([r[0] for r in rows],[r[2] for r in rows],'o-',label=f'read σ={read_e:g} e')
ax.set(xlabel='Jumlah phase steps',ylabel='Std galat fase [rad]',title='Eksperimen fixed total photoelectrons')
ax.legend();plt.show()
# Independently verify small-noise closed form using homoscedastic noise at four steps
noise_iid=.012
predicted=np.sqrt(2*noise_iid**2/(4*B0**2))
sim_iid=(A0+B0*np.cos(phi0+np.arange(4)*np.pi/2))[:,None]+rng_photon.normal(0,noise_iid,(4,nrep))
fit_iid=np.linalg.lstsq(np.column_stack([np.ones(4),np.cos(np.arange(4)*np.pi/2),np.sin(np.arange(4)*np.pi/2)]),sim_iid,rcond=None)[0]
phase_iid=np.arctan2(-fit_iid[2],fit_iid[1])
std_iid=float(np.std(np.angle(np.exp(1j*(phase_iid-phi0))),ddof=1))
print(f'Gaussian phase std: analytical={predicted:.4f}, Monte Carlo={std_iid:.4f} rad')
assert abs(std_iid/predicted-1)<0.10